# Neo4j Graph Database Tutorial

This notebook demonstrates connecting to Neo4j Aura, creating nodes, creating relationships, querying graph data, and deleting records.

## 1. Install the Neo4j driver

`%pip install neo4j` installs the official Python driver. The `%pip` form is preferred in Jupyter because it keeps the environment consistent with the active kernel.

In [ ]:
%pip install neo4j


## 2. Import `GraphDatabase`

`from package import name` imports a specific object from a Python package. `GraphDatabase` is the factory used to create a driver instance.

In [ ]:
from neo4j import GraphDatabase


## 3. Configure connection settings

The three variables contain the Aura URI, username, and password. `os.getenv()` reads values from environment variables, which is safer than hardcoding secrets in notebooks.

In [ ]:
import os

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

if not all([NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD]):
    raise ValueError("Set NEO4J_URI, NEO4J_USERNAME, and NEO4J_PASSWORD before running this notebook.")


## 4. Create a Neo4j driver

`GraphDatabase.driver()` creates the communication object used by Python to send Cypher queries to Neo4j. `auth=(username, password)` authenticates the connection.

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)


## 5. Verify connectivity

`verify_connectivity()` checks whether the driver can reach Neo4j and authenticate successfully. It raises an exception if anything is configured incorrectly.

In [ ]:
driver.verify_connectivity()


## 6. Create a reusable query helper

The function avoids repeating driver boilerplate. `parameters or {}` uses an empty dictionary when no parameters are passed.

In [ ]:
def run_query(query, parameters=None):
    records, _, _ = driver.execute_query(query, parameters_=parameters or {})
    return [record.data() for record in records]


## 7. Create one student node

The Cypher query creates a node with the `student` label and three properties. `s` is an alias for the node, and `RETURN s` sends the created node back.

In [ ]:
query = """
CREATE (s:student {name: $name, age: $age, city: $city})
RETURN s
"""
result = run_query(query, parameters={"name": "Rahul", "age": 24, "city": "Bengaluru"})
result


## 8. Reuse the query with different parameters

The same Cypher statement can create another student because values are passed separately from the query text.

In [ ]:
run_query(
    query,
    parameters={"name": "shadiya", "age": 24, "city": "BLR"}
)


## 9. Prepare student data for bulk insertion

`students` is a Python list containing dictionaries. Each dictionary represents one student.

In [ ]:
students = [
    {"student_id": "S001", "name": "Rahul", "age": 24, "city": "Bengaluru"},
    {"student_id": "S002", "name": "Priya", "age": 23, "city": "Pune"},
    {"student_id": "S003", "name": "Aman", "age": 25, "city": "Delhi"},
    {"student_id": "S004", "name": "Meera", "age": 22, "city": "Hyderabad"}
]


## 10. Create students in a loop

`CREATE` runs once per dictionary. The `for` loop assigns each dictionary to `student`, then passes it directly to the parameter map.

In [ ]:
query = """
CREATE (s:student {student_id: $student_id, name: $name, age: $age, city: $city})
RETURN s
"""
for student in students:
    run_query(query, parameters=student)


## 11. Prepare course data

Each dictionary contains a course identifier and name. The list will be processed in the next cell.

In [ ]:
courses = [
    {"course_id": "C001", "name": "Agentic AI"},
    {"course_id": "C002", "name": "LLM Engineering"},
    {"course_id": "C003", "name": "Graph Databases"}
]


## 12. Create course nodes

The loop executes the parameterized Cypher query for every course. `c` is the node alias and `course` is the dictionary value.

In [ ]:
query = """CREATE (c:course {course_id: $course_id, name: $name}) RETURN c"""
for course in courses:
    run_query(query, parameters=course)


## 13. Prepare skill names

`skills` is a list of strings. Each string will become the `name` property of a `skill` node.

In [ ]:
skills = [
    "Python",
    "Machine Learning",
    "RAG",
    "Neo4j",
    "FastAPI"
]


## 14. Create skill nodes

The loop passes a one-item dictionary because the query expects the `$name` parameter.

In [ ]:
query = """CREATE (s:skill {name: $name}) RETURN s"""
for skill in skills:
    run_query(query, parameters={"name": skill})


## 15. Prepare mentor data

The mentor is represented by one dictionary. `mentor_id` is a stable identifier that is preferable to using a name as the unique key.

In [ ]:
mentor = {
    "mentor_id": "M001",
    "name": "Sudhanshu",
    "company": "Euron"
}


## 16. Create a mentor node

This creates a node with the `mentor` label and three properties.

In [ ]:
query = """CREATE (m:mentor {mentor_id: $mentor_id, name: $name, company: $company}) RETURN m"""
run_query(query, parameters=mentor)


## 17. Create an enrollment relationship

The first two `MATCH` clauses find existing nodes. `MERGE` creates the relationship only when it does not already exist.

In [ ]:
query = """
MATCH (s:student {student_id: $student_id})
MATCH (c:course {course_id: $course_id})
MERGE (s)-[:ENROLLED_IN]->(c)
"""
run_query(query, parameters={"student_id": "S001", "course_id": "C001"})


## 18. Add properties to a relationship

`r` is an alias for the relationship, allowing `SET` to add or update relationship properties. `date_enrolled` captures context as metadata.

In [ ]:
query = """
MATCH (s:student {student_id: $student_id})
MATCH (c:course {course_id: $course_id})
MERGE (s)-[r:ENROLLED_IN]->(c)
SET r.date_enrolled = $date_enrolled
RETURN s, c, r
"""
run_query(query, parameters={"student_id": "S001", "course_id": "C001", "date_enrolled": "2026-09-20"})


## 19. Prepare multiple enrollments

Each dictionary maps a student ID to a course ID. `S005` is intentionally included as an example of a missing record or a future enrollment.

In [ ]:
enrollments = [
    {"student": "S001", "course": "C001"},
    {"student": "S002", "course": "C002"},
    {"student": "S003", "course": "C003"},
    {"student": "S005", "course": "C001"}
]


## 20. Create enrollments in a loop

For every enrollment, `MATCH` finds both endpoints and `MERGE` creates the edge. The nested dictionary approach keeps the query parameter map clean.

In [ ]:
query = """
MATCH (s:student {student_id: $student_id})
MATCH (c:course {course_id: $course_id})
MERGE (s)-[:ENROLLED_IN]->(c)
"""
for enrollment in enrollments:
    run_query(query, parameters={"student_id": enrollment["student"], "course_id": enrollment["course"]})


## 21. Create student-skill relationships

This models a many-to-many relationship: one student can have many skills, and one skill can belong to many students.

In [ ]:
student_skills = [
    {"student": "S001", "skill": "Python"},
    {"student": "S001", "skill": "RAG"},
    {"student": "S002", "skill": "Neo4j"},
    {"student": "S003", "skill": "Machine Learning"}
]

query = """
MATCH (s:student {student_id: $student_id})
MATCH (sk:skill {name: $skill})
MERGE (s)-[:HAS_SKILL]->(sk)
"""
for relation in student_skills:
    run_query(query, parameters={"student_id": relation["student"], "skill": relation["skill"]})


## 22. Traverse from a student to their skills

This is a read query. It starts at Rahul, follows outgoing `HAS_SKILL` relationships, and returns both the student and skill nodes.

In [ ]:
query = """
MATCH (s:student {name: $name})-[:HAS_SKILL]->(sk:skill)
RETURN s, sk
"""
run_query(query, parameters={"name": "Rahul"})


## 23. Filter students by age

`WHERE` filters rows matched by `MATCH`. `AND` requires both comparisons to be true. Returning properties instead of full nodes is simpler for reporting.

In [ ]:
query = """
MATCH (s:student)
WHERE s.age <= $age AND s.age >= $min_age
RETURN s.name, s.age
"""
run_query(query, parameters={"age": 24, "min_age": 20})


## 24. Delete a student and attached relationships

`DETACH DELETE` removes the matched node and all relationships connected to it. A plain `DELETE` would leave dangling relationships behind.

In [ ]:
query = """
MATCH (s:student {name: $name})
DETACH DELETE s
"""
run_query(query, parameters={"name": "shadiya"})


## Key syntax summary

- **Node:** `(n:Label {key: value})`
- **Relationship:** `(a)-[r:TYPE]->(b)`
- **Create:** `CREATE` always creates new nodes or relationships
- **Match:** `MATCH` locates existing graph elements
- **Merge:** `MERGE` avoids duplicates when inserting relationships
- **Delete:** `DETACH DELETE` removes a node and all attached edges